In [21]:
import jax.numpy as jnp
from scipy.stats import norm, multivariate_normal, binom
from IPython.display import Markdown, display
from jax import random

In [22]:
def to_latex_matrix(matrix):
    """Convert a JAX/numpy matrix to a LaTeX bmatrix string."""
    rows = []
    for row in matrix:
        rows.append(" & ".join(f"{val:.2f}" for val in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

def to_latex_vector(vector):
    """Convert a JAX/numpy vector to a LaTeX bmatrix column vector."""
    rows = r" \\ ".join(f"{val:.2f}" for val in vector)
    return r"\begin{bmatrix}" + rows + r"\end{bmatrix}"

# Gaussian Process Classification

## Dictionary


$\bf{X} = \text{Training points}$
$x^{*} = \text{Test points}$

Covariance matrix between training points:
$$\bf{K} = k(X,X)$$

Covariance matrix between test points:
$$k = k(x_{*},x_{*})$$

Covariance matrix between test points and training points:
$$\bf{k} = k(\bf{x_{*}},\bf{X})$$

In [46]:
#Define training data
xtrain = jnp.array([1,2,3])[:, None]

#Define training targets
ytrain = jnp.array([1,1,0])[:, None]

#Define test data
x_star = jnp.array([4])[:, None]

#Define laplace approximation
m = jnp.array([1.56, 0.84, -0.32])
S = jnp.array([[2.69, 1.46, 0.42],[1.46, 1.78, 1.2],[0.42, 1.2, 2.13]])

kernel = lambda x, xs: 5 * (1 + jnp.exp(-1/4 * (x-xs)**2))

def kernel_func(x,xs):
    return kernel(x.T,xs)

k = kernel_func(x_star, xtrain)
K = kernel_func(xtrain, xtrain)
Kstar = kernel_func(x_star, x_star)
h = jnp.linalg.solve(K, k) # k^T K^{-1}

#functions
probit = lambda x: norm.cdf(x)
sigmoid = lambda x: 1 / (1 + jnp.exp(-x))


## Prior predictive distribution - $p(f^{*} | x^{*})$

### Equation

$$p(f^{*} | x^{*}) = \mathcal{N}(f^{*} | 0, k_{1}(x^{*}, x^{*}))$$

### Code

In [24]:
prior_mu = 0
prior_var = kernel_func(x_star, x_star)

display(Markdown(rf"The prior predictive distribution is $p(f_* \mid x_*) = \mathcal{{N}}(f_* \mid \mu_*, \sigma_*^2)$ where $\mu_* = {prior_mu}$ and $\sigma_*^2 = {to_latex_matrix(prior_var)}$"))

The prior predictive distribution is $p(f_* \mid x_*) = \mathcal{N}(f_* \mid \mu_*, \sigma_*^2)$ where $\mu_* = 0$ and $\sigma_*^2 = \begin{bmatrix}10.00\end{bmatrix}$

## Prior predictive distribution - $p(y^{*} | x^{*})$

Prior predictive is always $0.5$ as long as the gaussian regression has a mean function of $\bf{0}$.

## Posterior distribution - $p(f^{*} | \bf{y}, x^{*})$

### Equation

$$= \mathcal{N}(f^* \mid \mu_{f^*}, \sigma^2_{f^*})$$
where
$$\mu_{f^*} = \mathbf{k}^T \mathbf{K}^{-1} \mathbf{m}$$
and
$$\sigma^2_{f^*} = k - \mathbf{k}^T \mathbf{K}^{-1} (\mathbf{K} - \mathbf{S}) \mathbf{K}^{-1} \mathbf{k}$$

### Code

In [ ]:
mu_f = m @ h
var_f =  Kstar - h.T @ ((K-S) @ h)

display(Markdown(rf"The posterior distribution is $p(f_* \mid \bf{{y}}, x_*) = \mathcal{{N}}(f_* \mid \mu_*, \sigma_*^2)$ where $\mu_* = {to_latex_vector(mu_f)}$ and $\sigma_*^2 = {to_latex_matrix(var_f)}$"))

The posterior distribution is $p(f_* \mid \bf{y}, x_*) = \mathcal{N}(f_* \mid \mu_*, \sigma_*^2)$ where $\mu_* = \begin{bmatrix}-0.71\end{bmatrix}$ and $\sigma_*^2 = \begin{bmatrix}4.16\end{bmatrix}$

## Posterior predictive distribution (*probit*) - $p(y^{*} | \bf{y}, x^{*})$

### Equation

$$p(y^{*} = 1 | \bf{y}, x^{*}) = \Phi\left(\frac{\mu_{f_{*}}}{\sqrt{\frac{8}{\pi} + \sigma_{f_{*}}^{2}}}\right)$$


Equations for $\mu_{f_{*}}$ and $\sigma^{2}_{f_{*}}$ can be found above.

### Code

In [45]:


p = probit(mu_f / (jnp.sqrt(8 / jnp.pi + var_f)))
display(Markdown(rf"The posterior predictive distribution is $p(y_* \mid \bf{{y}}, x_*) = {to_latex_matrix(p)}$ "))


The posterior predictive distribution is $p(y_* \mid \bf{y}, x_*) = \begin{bmatrix}0.39\end{bmatrix}$ 

## Posterior predictive distribution (*Monte Carlo*) - $p(y^{*} | \bf{y}, x^{*})$

### Equation

$$p(y^* = 1 \mid \mathbf{y}, x^*) \approx \frac{1}{S} \sum_{i=1}^{S} \sigma\left(f^{(i)}\right)$$

### Code

In [56]:
def mc_approx(num_samples,mu,sigma):
    key = random.PRNGKey(3)
    f = mu  + jnp.sqrt(sigma)*random.normal(key, shape=(num_samples, len(x_star)))
    p = sigmoid(f).mean(0)
    return p

p_mc = mc_approx(num_samples=1000, mu = mu_f, sigma = var_f)
display(Markdown(rf"The predictive probability is $p(y=1 \mid \bf{{x_*}}, y) = \frac{{1}}{{S}} \sum_{{i=1}}^{{S}} \sigma\left(f^{{(i)}}\right) = {to_latex_vector(p_mc)}$"))

The predictive probability is $p(y=1 \mid \bf{x_*}, y) = \frac{1}{S} \sum_{i=1}^{S} \sigma\left(f^{(i)}\right) = \begin{bmatrix}0.39\end{bmatrix}$